# Tetrahedral flux reconstruction and adaptive resolution

Canonical Raviart–Thomas reconstruction uses physical flux moments. The error estimator requires $k\geq\ell+d$, hence $k\geq\ell+3$ on tetrahedra. The analytical campaigns distinguish estimator behavior from pressure and vector-flux accuracy.

In [ ]:
import json
from pathlib import Path
import numpy as np
from threadpoolctl import threadpool_limits
from IPython.display import Image, display
from pymhm import TetraMesh, solve_darcy_3d, estimate_darcy_error_3d

root = Path.cwd()
if not (root / "examples").is_dir():
    root = root.parent
records = root / "examples/results/reconstruction3d"


## Represented nonzero boundary data

The affine exact pressure has constant physical flux $(-1,-2,-3)$. The local $P_3$ space, constant macroface flux and RT1 reconstruction satisfy the three-dimensional estimator hypotheses. Both field errors and all four estimator terms should be at the floating-point solve level.

In [ ]:
def pressure(x):
    """Affine pressure with exactly represented Dirichlet values."""
    return 1 + x @ np.array([1., 2., 3.])

with threadpool_limits(1):
    solution = solve_darcy_3d(TetraMesh.unit_cube(), degree=3, dirichlet=pressure)
    estimator = estimate_darcy_error_3d(solution, dirichlet=pressure)
    assert solution.l2_error(pressure) < 1e-11
    assert estimator.reconstruction.flux_l2_error([-1., -2., -3.]) < 1e-10
    assert estimator.total < 1e-10
    print("Pressure error:", solution.l2_error(pressure))
    print("Indicator:", estimator.total)


## Uniform and adaptive exact-field studies

The stored research runs use five uniform grids and eight adaptive states. Their error quadrature is checked independently. Effectivity compares the indicator with the broken energy error; it does not replace the physical pressure and flux errors.

In [ ]:
for name in ("uniform", "adaptive"):
    record = json.loads((records / f"{name}.json").read_text())
    assert record["local_degree"] >= record["trace_degree"] + 3
    assert record["source_changed_during_run"] is False
    assert len(record["rows"]) == (5 if name == "uniform" else 8)
    print(name, len(record["rows"]), "states")
    print([(r["macro_cells"], r["pressure_relative"], r["rt_flux_relative"], r["effectivity"]) for r in record["rows"]])
    display(Image(filename=str(root / f"docs/figures/reconstruction3d/{name}-convergence.png")))


## Separate local and skeletal approximation

All rows use the same macrogeometry and exact source. The study varies the local polynomial degree, then the flux trace degree and subface partition. Relative errors use full physical volume norms. In this experiment, $P_4/P_2$ is an approximation-space comparison; it is not assigned the estimator theorem, which would require local $P_5$ for that trace in three dimensions.

In [ ]:
record = json.loads((records / "resolution.json").read_text())
assert record["source_changed_during_run"] is False
for r in record["rows"]:
    print(r["name"], "pressure:", r["pressure_relative"], "raw flux:", r["raw_flux_relative"], "RT flux:", r["rt_flux_relative"])
control = json.loads((records / "reconstruction-order.json").read_text())
assert control["source_changed_during_run"] is False
print("Same pressure solution, RT3 relative flux error:", control["rt_flux_relative"])
for name in ("adaptive-components", "adaptive-profiles", "adaptive-profile-errors"):
    display(Image(filename=str(root / f"docs/figures/reconstruction3d/{name}.png")))
